# VLA 基础

## 0. VLA 的核心任务

VLA 是 **Vision-Language-Action model**，即视觉-语言-动作模型。它的核心任务是：

**给定视觉观察、机器人自身状态和自然语言指令，输出能够完成任务的机器人动作。**

最简形式是：

```text id="2zpa54"
输入：
    image / video observation
    robot state / proprioception
    language instruction

输出：
    robot action
```

可以写成：

$$
a_t = \pi_\theta(I_t, L, s_t)
$$

如果输出未来一段动作序列，则是：

$$
A_t = (a_t, a_{t+1}, \ldots, a_{t+H})
$$

$$
A_t \sim p_\theta(A_t \mid I_t, L, s_t)
$$

其中：

```text id="5i1wzz"
    I_t = 当前视觉观察
    L = 语言指令
    s_t = 机器人状态
    A_t = 未来一段动作 chunk
```

从更深层看，VLA 不只是“图像 + 文字 → 动作”的映射，而是试图学习一种 **多模态 affordance 层级的动作规划能力**：

```text id="vqeuus"
    看到物体
    理解语言目标
    判断物体的可操作性
    选择操作对象和部位
    生成短期动作轨迹
    通过低层控制器执行
```

例如：

```text id="mqtjgt"
指令：put the red cup into the blue bowl

VLA 需要理解：
1. red cup 是被操作对象
2. blue bowl 是目标容器
3. cup 可以被 grasp
4. bowl 可以作为 place-into target
5. 当前夹爪位置决定下一步动作
6. 输出靠近、抓取、搬运、释放等动作序列
```

所以 VLA 的本质不是单纯的视觉识别，也不是单纯的语言理解，而是：

**把视觉语义、语言语义、机器人身体状态和动作 affordance 连接起来的策略模型。**

---

# 1. VLA 的基本结构

一个基础 VLA 可以写成：

```text id="sxdq07"
    observation
        ↓
    modality-specific encoder
        ↓
    continuous embeddings
        ↓
    LLM / Transformer / multimodal backbone
        ↓
    hidden states / condition representation
        ↓
    action head
        ↓
    action
```

更具体地说：

```text id="ejzz3m"
    image
    ↓ vision encoder
    visual embeddings

    language instruction
    ↓ tokenizer + embedding layer
    text embeddings

    robot state
    ↓ MLP / state encoder
    state embeddings

    visual embeddings + text embeddings + state embeddings
    ↓ multimodal Transformer / VLA backbone
    condition representation / hidden states

    ↓ action head
    action vector / action token / action chunk
```

这里要特别注意一个术语问题：

**token 严格指离散符号或离散索引；embedding 是连续向量。**

文本中：

```text id="a8s57r"
"cup" → token id → embedding vector
```

图像中：

```text id="ev1slg"
image patch → vision encoder → visual feature → projector → visual embedding
```

所以论文里常说 “visual tokens”、“state tokens”，但更严谨地说，它们是：

```text id="dapn17"
token-like continuous embeddings
```

或者：

```text id="656got"
占据 Transformer 序列位置的连续模态表示
```

---

# 2. 输入数据如何进入 LLM / Transformer？

## 2.1 图像输入

图像不能直接输入 LLM。常见流程是：

```text id="wm0z03"
image
  ↓
patchify / vision encoder
  ↓
patch-level visual features
  ↓
projection layer
  ↓
visual embeddings
```

比如 ViT 会把图像切成 patch，每个 patch 得到一个向量。然后通过 projector 映射到和 LLM hidden size 相同的维度。

```text id="cfn0cu"
vision feature dim: 1024
LLM hidden dim: 4096

visual feature
  ↓ Linear projector
visual embedding with dim 4096
```

这些 visual embeddings 可以和 text embeddings 放到同一个 Transformer 序列中。

---

## 2.2 语言输入

语言指令通过 tokenizer 变成 token id，再通过 embedding table 变成连续向量：

```text id="31m1mg"
"pick up the red cup"
        ↓ tokenizer
token ids
        ↓ embedding layer
text embeddings
```

语言提供任务条件：

```text id="4klxfn"
要拿哪个物体？
要推、抓、放、打开，还是擦拭？
目标对象和目标位置是什么？
```

没有语言，机器人只看到世界；有了语言，机器人才知道当前任务目标。

---

## 2.3 机器人状态输入

机器人状态通常包括：

```text id="ctuukz"
    关节角 q
    关节速度 dq
    末端执行器位置
    末端执行器姿态
    夹爪开合状态
    力/触觉反馈
    历史动作
```

这些是连续数值，通常通过 MLP 编码：

```text id="t4nkk6"
    robot state
    ↓ normalization
    ↓ MLP / linear projector
    state embedding
```

如果输入历史状态序列，例如：

$$
s_{t-2},\ s_{t-1},\ s_t
$$

则可以得到 state embedding sequence：

$$
e^s_{t-2},\ e^s_{t-1},\ e^s_t
$$

state 的作用是告诉模型：

```text id="4qrkes"
机器人自己在哪里？
夹爪开着还是闭着？
是否已经靠近目标？
当前处于任务的哪个阶段？
```

只看图像是不够的，因为同一张图像下，机器人处于不同状态时，下一步动作可能完全不同。

---

# 3. 多模态 embedding 如何组织？

最简单的方式是拼接：

```text id="v4zf03"
    visual embeddings + state embeddings + text embeddings
            ↓
    Transformer self-attention
```

也可以用 special tokens 明示模态边界：

```text id="lqoxcu"
    <image>
    visual embeddings
    </image>

    <state>
    state embeddings
    </state>

    <instruction>
    text embeddings
    </instruction>
```

对于多相机输入，也可以加入 view embedding：

```text id="4d3r9v"
    <front_camera>
    front visual embeddings
    </front_camera>

    <wrist_camera>
    wrist visual embeddings
    </wrist_camera>
```

对于历史帧，可以加入 time embedding：

$$
I_{t-2},\ I_{t-1},\ I_t
$$

对于不同机器人，可以加入 embodiment embedding：

```text id="rz65pd"
    robot_type = Franka / UR5 / ALOHA / mobile manipulator
```

因此，LLM / Transformer 通常不只是接收一个“无标记的混合序列”，而是通过以下信息区分模态：

```text id="aw2fjv"
    special tokens
    modality embeddings
    position embeddings
    time embeddings
    camera-view embeddings
    robot embodiment embeddings
```

---

# 4. 输出是一个 modality channel 还是多个？

这取决于架构。

## 4.1 单通道统一序列

某些 VLA 会把动作也离散成 action tokens，于是文本 token 和动作 token 都在一个统一的序列里生成。

结构类似：

```text id="skzf2j"
    image + instruction
            ↓
           LLM
            ↓
    action tokens
```

优点是复用 LLM 的自回归框架。

缺点是动作本来是连续量，离散成 token 会带来精度损失和生成速度问题。

---

## 4.2 多头输出

也可以让 backbone 共享，但不同输出头负责不同任务：

```text id="r8q7gc"
shared VLA backbone
    ├── language head
    ├── action head
    ├── subtask prediction head
    ├── object detection / grounding head
```

这类结构更适合多任务共训练。

---

## 4.3 Flow / diffusion action head

现代连续控制 VLA 里，action head 也可能不是直接输出最终动作，而是输出生成过程中的修正方向。

结构是：

```text id="t9fn75"
    image + language + state
            ↓
    VLA / VLM backbone
            ↓
    condition representation c

    noisy action chunk + flow time τ + c
            ↓
    flow action head
            ↓
    velocity / denoising direction

    多步更新后得到：
    action chunk
```

---

# 5. 语义对齐：不同模态如何对齐到“某时刻的世界”？

VLA 的核心难点不是把不同 embedding 拼起来，而是让它们在语义和行为上对齐。

## 5.1 视觉-语言对齐

很多 VLA 不从零学习视觉-语言对齐，而是继承 VLM 的预训练能力。

VLM 已经在大量图文任务上学过：

```text id="6jwwxb"
图片中的 red cup ↔ 文本 "red cup"
图片中的 drawer ↔ 文本 "drawer"
图片中的 bowl ↔ 文本 "bowl"
```

常见训练任务包括：

```text id="ixkswu"
    image captioning
    VQA
    image-text matching
    contrastive learning
    object detection
    referring expression grounding
```

所以 VLA 通常从 VLM 开始，然后接上机器人动作数据，使模型从“看懂并说出来”变成“看懂并做出来”。

---

## 5.2 投影层对齐

图像 encoder 的输出空间和 LLM 的 hidden space 不同，因此需要 projector：

```text id="mwyf3q"
    visual feature
        ↓
    projection layer
        ↓
    LLM-compatible visual embedding
```

LLaVA 类模型就是典型例子：视觉 encoder 提供图像特征，projection layer 把视觉特征映射到语言模型可处理的 embedding 空间，然后和文本 embedding 一起进入 LLM。

不过要注意：

**投影到同一个 hidden dimension 不等于语义自动对齐。**

真正的对齐来自训练目标和数据分布。

---

## 5.3 Self-attention / cross-attention 对齐

拼接式输入依赖 self-attention：

```text id="q2ecot"
    visual embeddings + text embeddings + state embeddings
            ↓
    self-attention
```

这样文本中的 `"cup"` 可以 attend 到图像中的杯子区域，state embedding 可以和 visual embedding 对齐到当前夹爪状态。

另一种方式是 cross-attention：

```text id="hm9m7t"
    text/state/action query
            attends to
    visual key/value
```

这相当于让语言或动作表示主动读取视觉信息。

---

## 5.4 行为监督中的对齐

VLM 只能告诉模型：

```text id="0z9xnt"
    这是什么物体？
    它大概在哪里？
```

但 VLA 还要学：

```text id="dicfse"
    如何接近它？
    如何抓它？
    何时闭合夹爪？
    动作后会发生什么？
```

这部分来自机器人 demonstration。

训练样本通常是：

```text id="9ou7ss"
    image_t
    instruction
    robot_state_t
    expert_action_t or expert_action_chunk_t
```

模型通过模仿学习学到：

```text id="g0tz58"
语言目标 ↔ 视觉对象 ↔ 机器人状态 ↔ 动作结果
```

例如：

```text id="y2qtsn"
    "pick up the red cup"
    + 红杯子在左前方
    + 夹爪在右侧且打开
    → 向左前方移动，靠近杯子，闭合夹爪，抬起
```

这就是 VLA 中更深层的 grounding：

```text id="juiak1"
word ↔ object ↔ affordance ↔ action consequence
```

---

## 5.5 Auxiliary losses

为了避免模型只学到表面相关性，可以加入辅助损失：

```text id="ygyxrm"
    object detection loss
    bounding box grounding loss
    subtask prediction loss
    VQA / captioning loss
    image-text matching loss
    future state prediction loss
    action prediction loss
```

这些辅助任务帮助模型明示：

```text id="h8s0wq"
    目标物体在哪里？
    语言指代哪个对象？
    当前任务阶段是什么？
    下一步应该做哪个子任务？
    ...
```

---

# 6. 什么是 affordance？

Affordance 可以翻译成：

```text id="e1pp7s"
    可供性
    可操作性
    行动可能性
```

它指的是：

**物体或环境相对于某个行动者所提供的可能交互方式。**

例如：

```text id="rvw8if"
椅子：
    椅面 → 坐
    椅背 → 靠
    扶手 → 手扶

杯子：
    杯身 / 杯柄 → 抓
    杯口 / 内部空间 → 盛放
    杯子整体 → 移动 / 倒水

抽屉：
    把手 → 抓 / 拉
    抽屉空间 → 放入 / 取出

按钮：
    表面 → 按压
```

所以 affordance 不是单纯的物体类别，而是：

```text id="etlf2n"
    这个物体在当前状态下允许我做什么？
    哪个部位可以交互？
    采取动作后会发生什么？
```

对 VLA 来说，affordance 是连接语言和动作的关键中间层。

比如：

```text id="p8eia7"
    "open the drawer"
```

模型不能只识别 drawer，还要知道：

```text id="izxq44"
    drawer handle 是可抓 / 可拉部位
    抽屉有滑动方向
    打开动作需要先接近把手，再抓住，再向外拉
```

因此，VLA 的目标不是简单学习：

```text id="3sie5p"
    object label → action
```

而是学习：

```text id="2b2eb9"
    language instruction
        ↓
    visual object grounding
        ↓
    affordance recognition
        ↓
    local action planning
        ↓
    continuous control
```

---

# 7. VLA 会自回归吗？

答案是：

**可以自回归，也可以不自回归。**

关键取决于动作如何表示。

---

## 7.1 自回归动作 token 路线

如果把连续动作离散化成 action tokens，模型就可以像语言模型一样自回归生成：

```text id="xk41ry"
image + instruction + state
    → action_token_1
    → action_token_2
    → action_token_3
    → ...
```

形式类似：

$$
p(z_1, z_2, \ldots, z_N \mid c)
=
\prod_i p(z_i \mid z_{<i}, c)
$$

其中：

```text id="gxmdzm"
z_i = 动作 token
c = 条件变量，例如 image、language、state
```

优点：

```text id="xnrfsi"
    复用 LLM 框架
    训练目标简单
    可以把动作和语言放进统一 token 序列
```

缺点：

```text id="v85s36"
    连续动作被离散化
    动作精度受 binning 影响
    必须先生成前一个 token 才能生成下一个
    推理速度受限
    高频控制不自然
```

这种生成方式像 RNN 一样有时序约束。虽然用的是 Transformer，但生成过程仍然是 sequential decoding，不能一次性并行得到完整动作序列。

---

## 7.2 非自回归 action chunk 路线

另一类方法直接生成未来一段动作：

$$
A_t = (a_t, a_{t+1}, \ldots, a_{t+H})
$$

这叫 action chunking。

优点：

```text id="5x8r9y"
动作更平滑
可以表达短期运动意图
减少单步行为克隆的抖动
适合连续控制
可以降低大模型高频推理压力
```

缺点：

```text id="dw09q7"
越远的动作越不可信
长 horizon 容易漂移
接触发生后后续动作可能失效
真实执行时通常只用前几步
```

所以 action chunk 不是“完整开放环轨迹”，而是：

```text id="sfie8y"
短期局部轨迹 + 意图表示
```

---

# 8. Flow matching / diffusion policy 如何生成 action chunk？

## 8.1 直接 action head

普通 VLA 可以直接输出 action chunk：

```text id="ppx0mo"
    condition c
        ↓
    action head
        ↓
    A_t
```

公式：

$$
A_t = f_\theta(c)
$$

其中：

```text id="epijwb"
c = VLA backbone 编码出的 condition representation
```

训练时做 MSE：

$$
\mathcal{L} = \left| A_{\mathrm{pred}} - A_{\mathrm{expert}} \right|^2
$$

这种方式简单，但容易学到平均动作。

---

## 8.2 Flow matching action head

Flow matching 不直接输出最终动作，而是学习：

```text id="73xf7g"
如何把 noisy action chunk 修正成合理 action chunk
```

结构是：

```text id="bw5fpl"
    image + language + state
            ↓
    VLA backbone
            ↓
    condition c

    noisy action chunk X_τ
    + flow time τ
    + condition c
            ↓
    flow action head
            ↓
    velocity vθ

    多步积分
            ↓
    action chunk A
```

公式是：

$$
\frac{dX_\tau}{d\tau}
=
v_\theta(X_\tau, \tau, c)
$$

这里：

```text id="r6ar3h"
X_τ = 当前去噪过程中的 noisy action chunk
τ = flow / denoising time
c = image-language-state condition
vθ = 修正方向 / velocity field
```

最终：

```text id="e1j7ke"
X_0 = random noise action chunk
X_1 = generated action chunk
```

所以在 flow matching 版 VLA 中：

```text id="3hm0ai"
    VLA backbone 主要输出 condition representation
    flow action head 负责在这个 condition 下生成连续动作
```

整个模型最终仍然输出 action chunk，但内部不是：

```text id="no92da"
    condition → action
```

而是：

```text id="gvfom0"
    condition + noise → action distribution
```

---

## 8.3 噪声到底是什么？

噪声不是传感器噪声，也不是机器人抖动。

在 VLA 的 action diffusion / flow 中，噪声是：

```text id="wewbnf"
和 action chunk 同形状的随机数组
```

如果真实动作是：

$$
A \in \mathbb{R}^{H \times d_a}
$$

那么噪声也是：

$$
Z \in \mathbb{R}^{H \times d_a}
$$

其中：

```text id="3c5rur"
H = action horizon
d_a = action dimension
```

区别是：

```text id="na17k0"
真实动作来自专家 demonstration 分布
噪声来自标准高斯分布
```

训练时，模型学习如何把噪声分布变成真实动作分布。

---

## 8.4 为什么从噪声开始，而不是从粗糙轨迹开始？

从噪声开始的优点是：

```text id="7ef91k"
起点简单
不依赖外部 planner
可以表达多峰分布
可以从不同噪声生成多条候选轨迹
```

例如：

```text id="voias0"
z1 → 从左侧抓
z2 → 从右侧抓
z3 → 从上方抓
```

如果从粗糙轨迹开始，则更像：

```text id="nn31nz"
trajectory refinement
guided diffusion
residual correction
```

这在工程上当然可以，而且很合理。但它需要另一个系统先给出粗轨迹，例如 motion planner、heuristic 或上一轮 action chunk。

所以：

```text id="mhbq8t"
纯噪声起点：
    适合完整建模 p(action_chunk | condition)

粗糙轨迹起点：
    适合工程上的 warm start / refinement
```

---

# 9. 一长串动作里能用几步？

训练时模型可能输出：

$$
H = 16,\ 32,\ 64
$$

但真实部署时，通常不会完整执行整段动作，而是：

```text id="lmhadg"
预测未来 H 步
只执行前 k 步
重新观察
再次预测
```

这叫 receding horizon control。

例如：

```text id="6x7ca4"
t = 0:
    预测 a_0 到 a_15
    执行 a_0 到 a_3

t = 4:
    重新观察
    预测 a_4 到 a_19
    执行 a_4 到 a_7
```

通常：

```text id="3ka2q9"
    前几步：可执行动作
    中间几步：短期轨迹参考
    后面几步：意图 / 趋势
```

越靠后，越容易受执行误差、接触变化、遮挡、物体滑动等影响。

所以 action chunk 的作用不是让机器人开放环执行到底，而是：

```text id="w882wn"
    让模型学到短期计划性和轨迹平滑性
    同时通过频繁重规划保持闭环纠错
```

---

# 10. VLA 的控制精度如何？

这要分层看。

如果把 VLA 当成：

```text id="7j4i4b"
    毫米级底层控制器
```

那它目前很难胜过传统控制、运动规划、视觉伺服、力控、阻抗控制或专用 RL policy。

但如果把 VLA 当成：

```text id="jo9xuv"
    affordance 层级的高维语义-动作规划器
```

它是有价值的。

VLA 擅长：

```text id="0hrb4o"
    根据语言选择目标物体
    理解任务语义
    在多物体环境中决定操作对象
    从示范中学习常见操作模式
    生成粗到中等精度的局部动作
```

VLA 不擅长：

```text id="tpe2ew"
    毫米级插孔
    高速动态控制
    强接触力控
    严格稳定性要求
    高精度工业装配
```

主要误差来源包括：

```text id="2xjown"
    视觉定位误差
    语言 grounding 错误
    动作表示误差
    坐标系不一致
    动作离散化误差
    horizon 漂移
    接触反馈不足
    训练数据分布偏差
    LLM / VLM 语义幻觉或推理错误
```

LLM 会算错四则运算，说明它不是严格符号计算器。同样，VLA 也不是严格控制器。它的强项不是“精确计算每一个控制量”，而是：

```text id="0zw8bm"
    从多模态上下文中生成一个大致合理的动作分布
```

真正可靠的系统通常需要：

```text id="9rgfop"
    VLA / VLM：理解任务和 affordance
    motion planner：保证几何可行和避障
    low-level controller：保证执行稳定、限速、力控
    visual servo / tactile feedback：做精细闭环修正
```

---

# 11. 动作规划到底由谁负责？

VLA 中的“规划”可以分成多层：

```text id="3azf3k"
    高层任务规划：
        做什么，先后顺序是什么

    中层 affordance / skill 规划：
        操作哪个物体，哪个部位，下一阶段目标是什么

    低层局部轨迹生成：
        未来 H 步 dx dy dz / rotation / gripper 怎么动

    底层控制执行：
        IK、关节控制、力控、阻抗控制、安全约束
```

Flow matching 或 diffusion policy 主要负责：

```text id="2xaetp"
    低层局部 action chunk generation
```

VLA backbone 负责：

```text id="6e9i7e"
    理解当前场景、语言指令和机器人状态
    提供 condition representation
```

高层 LLM / VLM planner 或 task policy 负责：

```text id="owax8x"
    长时程任务分解
    下一步子任务选择
    失败后的重规划
```

传统 motion planner / controller 负责：

```text id="bkjh00"
    可达性
    避障
    速度 / 加速度约束
    力控
    安全执行
```

所以最可靠的理解不是：

```text id="qc6vpn"
    VLA = 全部规划 + 全部控制
```

而是：

```text id="erf2oh"
    VLA = 多模态语义理解 + affordance 判断 + 局部动作策略
```

---

# 12. 泛用性如何保证？

VLA 的泛化能力主要来自几个方面。

## 12.1 VLM 预训练

VLM 继承互联网图文知识，能识别大量物体、场景和语言表达。

这使 VLA 不完全依赖机器人数据从零学习所有语义。

---

## 12.2 大规模机器人示范数据

机器人数据提供：

```text id="xy5mkq"
    动作如何执行
    物体如何接触
    夹爪什么时候闭合
    任务阶段如何推进
```

这部分能力单靠图文预训练无法获得，必须来自真实或仿真的机器人操作数据。

---

## 12.3 异构数据共训练

VLA 可以通过多来源数据提升泛化能力，例如：

```text id="eh4rzs"
    多机器人数据
    多任务 demonstration
    web 图文数据
    object detection 数据
    subtask prediction 数据
    语言指导数据
    失败恢复数据
```

这说明泛化不是单靠模型结构，而是靠：

```text id="l4pkyv"
    数据多样性
    任务多样性
    模态监督多样性
    环境多样性
```

---

## 12.4 避免伪相关

如果训练集中总是：

```text id="5zc2b2"
    红杯子在左边
    蓝碗在右边
```

模型可能学到：

```text id="k1l98q"
    red cup = 左边那个物体
```

而不是真正 grounding 到红色杯子。

所以需要：

```text id="zs1q1f"
    物体位置随机化
    颜色 / 形状 / 位置解耦
    语言表达多样化
    反事实样本
    遮挡和光照变化
    多视角输入
    失败恢复数据
```

这和 ROP 的思想很接近：

**如果表面顺序固定，智能体可能学捷径；如果打乱观察顺序或解耦属性，才更可能学到真正的概念对应。**

---

# 13. 最终总结

VLA 可以理解为：

**一种以视觉、语言和机器人状态为条件的具身策略模型。**

它的核心不是让 LLM 直接精确计算电机控制量，而是让模型学习：

```text id="lmk6hb"
    语言目标
        ↓
    视觉 grounding
        ↓
    物体 affordance
        ↓
    任务阶段
        ↓
    局部动作轨迹
```

因此，VLA 最适合被看作：

```text id="ggra68"
    affordance-level semantic action planner
    +
    local visuomotor policy
```

而不是：

```text id="s0xrbr"
    完全替代传统控制算法的万能底层控制器
```

在 flow matching / diffusion policy 架构中，VLA backbone 更像 condition encoder：

```text id="lvu98g"
    image + language + state
            ↓
    condition representation
```

动作生成器则根据这个 condition，把噪声 action chunk 逐步修正成合理的连续动作轨迹：

```text id="q9z4ej"
    noise action chunk
    + condition
            ↓
    flow / diffusion action head
            ↓
    action chunk
```

真实执行时通常只执行 action chunk 的前几步，然后重新观察、重新生成，以保持闭环控制。

所以 VLA 的合理定位是：

```text id="8o0u7m"
    高层语义理解
    + affordance 判断
    + 短期动作生成
    + 低层控制器闭环执行
```

这也是为什么 VLA 在机器人领域重要：
它不是因为能精确替代传统控制，而是因为它提供了一种把 **语言、视觉、世界语义、物体可操作性和机器人动作** 统一到同一个学习框架中的方法。

---